# L4 · Notebook 02 — Puzzle：估值不准时贪心也能输

**对应 PPT**：`L4-DP.tex` ★ Puzzle 帧（line 544）

## 教学目标

策略改进定理只保证：对\textbf{真实} $V^\pi$ 贪心 $\Rightarrow V^{\pi'} \ge V^\pi$。

但如果你拿\textbf{错的} $\widetilde V \ne V^\pi$ 来贪心，结果可以\textbf{任意糟糕}。

本 notebook 用 2-state MDP 反例 + GridWorld 实测两个角度验证：
1. 2-状态 MDP（PPT line 552 反例）：扫描 $\widetilde V(s_1)$ 误差，看贪心何时翻车
2. 3×3 GridWorld：PE 截断到第 $k$ 步后贪心改进，看 $k$ 多小时策略错

## 结论

**PI 内层 PE 必须收敛，否则贪心改进可能选错动作**——这正是 PI 每轮代价高的根源。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

## 1. 2-state MDP 反例（PPT line 552）

$\mathcal S = \{s_1, s_2\}$，$\mathcal A = \{a, b\}$，$\gamma = 0.9$。

- $s_1 \xrightarrow{a} s_1$，$r=0$
- $s_1 \xrightarrow{b} s_2$，$r=-1$
- $s_2 \xrightarrow{\cdot} s_2$，$r=+10$

**真实价值**（当前策略 $\pi(s_1)=a$）：$V^\pi(s_1)=0$，$V^\pi(s_2)=10/(1-0.9)=100$。

**真正的最优策略**：$\pi^*(s_1)=b$（『出去』），$V^{\pi^*}(s_1) = -1 + 0.9 \cdot 100 = 89$。

所以从 $\pi$ 改进到 $\pi^*$ 应该把 $V(s_1)$ 从 $0$ 拉到 $89$（**严格更优**）。

In [ ]:
gamma = 0.9

def Q_from_V_hat(V_hat_s1, V_hat_s2):
    """基于错误估计 V_hat 算 Q(s1, ·)，看贪心选什么。"""
    Q_s1_a = 0 + gamma * V_hat_s1   # 留在 s1
    Q_s1_b = -1 + gamma * V_hat_s2  # 去 s2
    return Q_s1_a, Q_s1_b

# 三种估值情况
scenarios = [
    ('真实 V^π', 0, 100),
    ('V̂(s2)=0 (PE 早停)', 0, 0),
    ('V̂(s2)=-100 (符号错)', 0, -100),
    ('V̂(s1)=200 高估 s1', 200, 100),
]

print(f'{"场景":<25s} | Q(s1,a)  Q(s1,b)  | 贪心 | 是否最优')
print('-' * 75)
for name, v1, v2 in scenarios:
    qa, qb = Q_from_V_hat(v1, v2)
    greedy = 'b' if qb > qa else 'a'
    is_opt = '✓' if greedy == 'b' else '✗'
    print(f'{name:<25s} | {qa:+6.2f}  {qb:+6.2f}  |  {greedy}   |  {is_opt}')

**观察**：
- 真实 $V^\pi$ → 贪心正确选 $b$
- $V(s_2)$ 估低（甚至 $-100$）→ 贪心仍选 $a$（保守），错过 $b$
- $V(s_1)$ 高估 200 → 贪心 $Q(s_1,a)=180$ vs $Q(s_1,b)=89$，**严重错失最优**

**根源**：贪心是"信任 $\widetilde V$ + 一步前瞻"——$\widetilde V$ 错则贪心错。

## 2. 扫描 $\widetilde V(s_2)$ 误差，看贪心翻转

固定 $\widetilde V(s_1)=0$，扫描 $\widetilde V(s_2)$ 从 $-200$ 到 $+200$；找贪心翻为 $b$ 的临界点。

In [ ]:
v2_range = np.linspace(-200, 200, 200)
greedy_b = []
for v2 in v2_range:
    qa, qb = Q_from_V_hat(0, v2)
    greedy_b.append(qb > qa)
greedy_b = np.array(greedy_b)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(v2_range, [0 + gamma*0 for _ in v2_range], '-', label='Q(s1,a) = 0+γ·V(s1) = 0', color='C0')
ax.plot(v2_range, [-1 + gamma*v2 for v2 in v2_range], '-', label='Q(s1,b) = −1+γ·V(s2)', color='C1')
ax.axvline(100, color='green', linestyle='--', alpha=0.5, label='真值 V^π(s2)=100')
ax.axvline(1/gamma, color='red', linestyle=':', alpha=0.7, label=f'临界 V(s2)=1/γ≈{1/gamma:.2f}')
ax.axhline(0, color='gray', linewidth=0.5)
ax.set_xlabel(r'估计 $\widetilde V(s_2)$')
ax.set_ylabel(r'$Q(s_1, \cdot)$')
ax.set_title('贪心翻转：$\\widetilde V(s_2) > 1/\\gamma \\approx 1.11$ 时才会选 b')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('figures/puzzle_v2_sweep.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'临界值：V̂(s2) 必须 > 1/γ = {1/gamma:.4f} 才能让贪心选 b（正确）')
print(f'真值 V^π(s2) = 100 远超临界值，所以"评估准"时贪心正确')

## 3. GridWorld 实测：PE 截断 $k$ 步后贪心

3×3 网格，初始策略 $\pi_0$ = 全部 "stay"（最差策略 $V^{\pi_0} \equiv 0$）。

对 $\pi_0$ 做 $k$ 步 PE 得到 $V_k$，然后贪心改进得 $\pi'_k$。

记录 $V^{\pi'_k}$（真值），看 $k$ 多小时改进策略效果差。

In [ ]:
# 复用 L1 GridWorld
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'L1-mdp-foundations')))
from grid_world import ACTIONS, GridConfig, GridWorld

env = GridWorld(GridConfig(gamma=0.9))
states = list(env.all_states())
n = len(states)
idx = {s: i for i, s in enumerate(states)}

def pe_truncated(env, policy, k_steps, V0=None):
    """对 policy 做 k_steps 次 PE（同步）；V0=None 时从 0 起。"""
    V = np.zeros(n) if V0 is None else V0.copy()
    for _ in range(k_steps):
        V_new = np.zeros_like(V)
        for i, s in enumerate(states):
            a = policy[s]
            s_next, r, _ = env.step(s, a)
            V_new[i] = r + env.cfg.gamma * V[idx[s_next]]
        V = V_new
    return V

def greedy_policy(env, V):
    pol = {}
    for i, s in enumerate(states):
        best_q, best_a = -np.inf, ACTIONS[0]
        for a in ACTIONS:
            s_next, r, _ = env.step(s, a)
            q = r + env.cfg.gamma * V[idx[s_next]]
            if q > best_q:
                best_q, best_a = q, a
        pol[s] = best_a
    return pol

def true_value(env, policy, tol=1e-10):
    """PE 跑到收敛得真值 V^π。"""
    return pe_truncated(env, policy, k_steps=2000)

# 初始策略 π_0：全部 stay
pi_0 = {s: 'stay' for s in states}
V_pi_0 = true_value(env, pi_0)
print(f'V^π_0 (全 stay) 累计: {V_pi_0.sum():.3f}')

# 不同 PE 截断步数下的改进策略
results = []
for k in [1, 2, 3, 5, 10, 50, 200]:
    V_k = pe_truncated(env, pi_0, k_steps=k)
    pi_k = greedy_policy(env, V_k)
    V_pi_k_true = true_value(env, pi_k)
    results.append((k, V_pi_k_true.sum()))
    print(f'PE 截断 k={k:3d}: 改进策略真值累计 = {V_pi_k_true.sum():>7.3f}')

V_optimal = true_value(env, greedy_policy(env, true_value(env, pi_0)))
print(f'\n参考：完整 PE + 改进的真值累计 = {V_optimal.sum():.3f}')

In [ ]:
ks, vals = zip(*results)
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogx(ks, vals, 'o-', linewidth=2)
ax.axhline(V_optimal.sum(), color='red', linestyle='--', label='完整 PE + 改进 (上限)')
ax.axhline(V_pi_0.sum(), color='gray', linestyle=':', label='原策略 π_0 (全 stay)')
ax.set_xlabel('PE 截断步数 k')
ax.set_ylabel('改进后策略的真值累计 $\\sum_s V^{\\pi_k}(s)$')
ax.set_title('PE 截断步数 vs 贪心改进质量')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('figures/puzzle_pe_truncation.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. 课堂诊断小结

| 论断 | 数值证据 |
|---|---|
| 2-state 反例：$\widetilde V$ 错则贪心错 | $\widetilde V(s_2)<1/\gamma$ 时贪心选 $a$，错过最优 $b$ |
| GridWorld：PE 截断 1-2 步 → 改进策略差 | $k=1$ 时显著低于 $k=50+$ |
| PE 跑得越久，贪心改进质量越好 | 单调上升直到收敛上限 |

## 工程意义

- **PI** 内层 PE 必须跑到收敛——这是 PI 每轮代价高的根源
- **Modified PI**（截断 PI）每轮只评 $m$ 步，靠**反复迭代**摊掉评估误差
- **L7 Q-learning / L8 DQN**：用近似 $\widetilde Q_\theta$ 做 $\arg\max_a$——这正是 deadly triad 与训练发散的根源

## 思考题

1. 把初始策略 $\pi_0$ 改成"全 right"（不同于 stay 的策略），$k=1$ 的改进策略是否还差？
2. 引入 PE 衰减步长 $\alpha_k$（不是直接 $T^\pi$ 替换），收敛是否更稳？这对应 L5 RM 的什么场景？
3. 若把 $\gamma$ 改到 $0.99$，PE 截断步数需要多大才让改进策略接近最优？